## Notebook for identifying TF communities in the Norman dataset

In [1]:
# set up
import numpy as np
import pandas as pd
import math
from scipy.stats import false_discovery_control
import pickle
import gseapy as gp

# Import GRN analysis functions from gcrl package
from gcrl.grn import (
    build_tf_tf_regulatory_layer,
    build_tf_tf_cotarget_layer,
    sweep_hyperparams,
    summarize_all_stabilities,
    build_coassociation_matrix,
    consensus_partition_from_coassoc,
    community_stats,
    run_single_layer_leiden_reg,
    run_single_layer_leiden_cot,
    run_ora_for_clusters,
    filter_gene_sets_by_size_and_level,
    build_cluster_gene_sets,
    extract_go_ids_from_terms,
    compute_go_levels
)

In [2]:
# control parameters
top_percentage = 0.50  # percentage of top edges to keep in the final GRN
data_folder = '../../../data/real/Norman2019/'  # path to data folder
significance_level = 0.05  # significance level for FDR correction

### Loading and filtering GRN

In [3]:
#Load GRN 
raw_grn = pd.read_csv(data_folder + 'GRN/' + "raw_GRN.csv", index_col=0)

#Drop NA values in p-value column
raw_grn = raw_grn.dropna(subset=["p"])

#FDR filtering and only leaving adj p-values values less than 0.05
raw_grn.loc[:, 'adjp'] = false_discovery_control(raw_grn['p'].values, method='bh')
filtered_grn = raw_grn[raw_grn['adjp'] <= significance_level].copy()

# Sort by absolute coefficient descending
filtered_grn = filtered_grn.sort_values(by='coef_abs', ascending=False).reset_index(drop=True)

# selecting top_percentage of top edges
num_top_edges = math.ceil(len(filtered_grn) * top_percentage)
current_grn = filtered_grn.iloc[:num_top_edges, :]

# print statistics
print(current_grn['source'].nunique(), current_grn['target'].nunique())
current_grn


141 2048


,source,target,coef_mean,coef_abs,p,-logp,adjp
0,JUND,HIST1H4C,0.801887,0.801887,6.427243e-13,12.191975,6.133731e-12
1,PITX1,HIST1H4C,0.566257,0.566257,7.473443e-08,7.126479,1.766064e-07
2,KLF1,HBZ,0.548236,0.548236,8.472660e-20,19.071980,2.997454e-17
3,LYL1,HSP90AA1,0.462909,0.462909,1.499360e-25,24.824094,1.036404e-21
4,ENO1,RAN,0.459944,0.459944,7.764523e-19,18.109885,1.465798e-16
...,...,...,...,...,...,...,...
38349,PRDM1,MVP,-0.000333,0.000333,5.406043e-14,13.267121,7.915708e-13
38350,FOS,LINC00987,-0.000333,0.000333,9.313351e-06,5.030894,1.597743e-05
38351,MEF2C,SHC4,-0.000333,0.000333,1.662021e-10,9.779363,7.130883e-10
38352,KLF2,SLC16A3,0.000333,0.000333,1.868734e-08,7.728453,4.961866e-08


### Retrieving the list of TFs

In [4]:
# Load TF reference
tf_info = pd.read_parquet(data_folder + 'GRN/' + "hg38_TFinfo_dataframe_gimmemotifsv5_fpr2_threshold_10_20210630.parquet") ## Google Drive: gCAL_data > Norman_results
tf_names = list(tf_info.columns.unique())
tf_names = [item for item in tf_names if item in current_grn['source'].unique() or item in current_grn['target'].unique()]
print(f"Number of TFs in reference: {len(tf_names)}")

Number of TFs in reference: 141


## Initial assessments

In [5]:
# 1. Build layers
g_reg, tf_index = build_tf_tf_regulatory_layer(current_grn, tf_names)
g_cot, sim_matrix = build_tf_tf_cotarget_layer(current_grn, tf_names, tf_index, min_similarity=0.15)
print('Number of edges in g_reg: ', g_reg.ecount())
print('Number of edges in g_cot: ', g_cot.ecount())

# 2. computing memberships
mem_reg, _, _ = run_single_layer_leiden_reg(g_reg, gamma_reg=1, seed=0)
mem_cot, _, _ = run_single_layer_leiden_cot(g_cot, gamma_cot=1.2, seed=0)

# 3. computing community statistics
print(community_stats(g_reg, mem_reg).head(n=10))
print(community_stats(g_reg, mem_reg).shape)
print(community_stats(g_cot, mem_cot).head(n=10))
print(community_stats(g_cot, mem_cot).shape)


Number of edges in g_reg:  2798
Number of edges in g_cot:  2787
   community  size  w_internal   density
2          0    34    1.548721  0.002761
3          1    27    1.608759  0.004583
4          2    25    1.627376  0.005425
0          3    21    0.526613  0.002508
1          4    19    1.077583  0.006302
5          5    13    0.701937  0.008999
6          6     1    0.000000  0.000000
7          7     1    0.000000  0.000000
(8, 4)
   community  size  w_internal   density
1          0    31   31.062353  0.066801
4          1    23   53.919858  0.213122
5          2    21   21.858653  0.104089
6          3    18   60.977836  0.398548
3          4    17   27.209557  0.200070
0          6    15   18.569932  0.176856
2          5    15   20.014301  0.190612
7          7     1    0.000000  0.000000
(8, 4)


In [ ]:

# 2. Hyperparameter grids (tune as needed)
gamma_reg_list = [1, 1.1, 1.2]
gamma_cot_list = [1, 1.1, 1.2]
cot_weight_list = [0.1, 0.5, 1.0]   # plays the role of your "ω"

# 3. Sweep & collect partitions
results = sweep_hyperparams(
    g_reg,
    g_cot,
    gamma_reg_list=gamma_reg_list,
    gamma_cot_list=gamma_cot_list,
    cot_weight_list=cot_weight_list,
    n_seeds=100,
    n_iterations=-1,   # until convergence
)

Done γ_reg=1, γ_cot=1, w_cot=0.1
Done γ_reg=1, γ_cot=1, w_cot=0.5
Done γ_reg=1, γ_cot=1, w_cot=1.0
Done γ_reg=1, γ_cot=1.1, w_cot=0.1
Done γ_reg=1, γ_cot=1.1, w_cot=0.5
Done γ_reg=1, γ_cot=1.1, w_cot=1.0
Done γ_reg=1, γ_cot=1.2, w_cot=0.1
Done γ_reg=1, γ_cot=1.2, w_cot=0.5
Done γ_reg=1, γ_cot=1.2, w_cot=1.0
Done γ_reg=1.1, γ_cot=1, w_cot=0.1
Done γ_reg=1.1, γ_cot=1, w_cot=0.5
Done γ_reg=1.1, γ_cot=1, w_cot=1.0
Done γ_reg=1.1, γ_cot=1.1, w_cot=0.1
Done γ_reg=1.1, γ_cot=1.1, w_cot=0.5
Done γ_reg=1.1, γ_cot=1.1, w_cot=1.0
Done γ_reg=1.1, γ_cot=1.2, w_cot=0.1
Done γ_reg=1.1, γ_cot=1.2, w_cot=0.5
Done γ_reg=1.1, γ_cot=1.2, w_cot=1.0
Done γ_reg=1.2, γ_cot=1, w_cot=0.1
Done γ_reg=1.2, γ_cot=1, w_cot=0.5
Done γ_reg=1.2, γ_cot=1, w_cot=1.0
Done γ_reg=1.2, γ_cot=1.1, w_cot=0.1
Done γ_reg=1.2, γ_cot=1.1, w_cot=0.5
Done γ_reg=1.2, γ_cot=1.1, w_cot=1.0
Done γ_reg=1.2, γ_cot=1.2, w_cot=0.1
Done γ_reg=1.2, γ_cot=1.2, w_cot=0.5
Done γ_reg=1.2, γ_cot=1.2, w_cot=1.0


In [7]:

# 4. Stability summaries
stability = summarize_all_stabilities(
    results,
    compute_ari=True,   # no sklearn, no warnings, faster
    max_pairs=None        # or None if you want all pairs
)
#stability  # dict: (gamma_reg, gamma_cot, omega) -> ARI/VI summary

In [8]:
# saving / loading
#with open("multiplex_results.pkl", "wb") as f:
#    pickle.dump(results, f)
#with open("multiplex_stability.pkl", "wb") as f:
#    pickle.dump(stability, f)
#with open("multiplex_results.pkl", "rb") as f:
#    results = pickle.load(f)
#with open("multiplex_stability.pkl", "rb") as f:
#    stability = pickle.load(f)


In [9]:
# Pick the most stable triple
best_key = max(stability.items(), key=lambda kv: kv[1]['ARI_mean'])[0]
print("Best hyperparams:", best_key)
print("Best stability stats:", stability[best_key])

# 5. Consensus clustering for best hyperparams
memberships_list = results[best_key]
C = build_coassociation_matrix(memberships_list)
consensus_labels, consensus_graph = consensus_partition_from_coassoc(
    C, min_coassoc=0.5, resolution=1
)

# 6. Diagnostics per layer
df_reg_stats = community_stats(g_reg, consensus_labels)
df_cot_stats = community_stats(g_cot, consensus_labels)

print("Regulatory layer community stats:")
print(df_reg_stats)

print("Co-target layer community stats:")
print(df_cot_stats)


Best hyperparams: (1.2, 1, 0.1)
Best stability stats: {'VI_mean': np.float64(0.8174111254869254), 'VI_std': np.float64(0.3719683068069459), 'ARI_mean': np.float64(0.7146675060751068), 'ARI_std': np.float64(0.14947411359622254), 'n_pairs': 4950}
Regulatory layer community stats:
   community  size  w_internal   density
0          0    39    2.384346  0.003218
1          1    31    0.491486  0.001057
3          2    29    0.244104  0.000601
2          4    21    1.713199  0.008158
4          3    21    0.980276  0.004668
Co-target layer community stats:
   community  size  w_internal   density
0          0    39  148.426706  0.200306
1          1    31   31.348231  0.067416
3          2    29   37.205673  0.091640
2          4    21   36.908616  0.175755
4          3    21   73.235022  0.348738


## Gene set enrichment analysis

In [10]:
# 1) Build per-cluster gene sets (TFs + targets)
cluster2genes = build_cluster_gene_sets(
    tf_names=tf_names,
    consensus_labels=consensus_labels,
    current_grn=current_grn,
)

# 2) Load GO BP gene sets (Homo sapiens)
raw_go_bp = gp.get_library(name='GO_Biological_Process_2025', organism='Human')

# 2.1) Extract GO IDs present in this library
go_ids_in_library = extract_go_ids_from_terms(raw_go_bp)

# 2.2) Compute levels only for these GO IDs (BP namespace)
# Note: go_obo_path is optional - defaults to gCRL/data/reference/ontologies/go-basic.obo
go_term_levels = compute_go_levels(
    go_ids=go_ids_in_library,
    namespace="biological_process",
)

Loading GO DAG from /home/laganiv/Desktop/projects/CausalEmbed/grn_crl/gCAL/gCRL/data/reference/ontologies/go-basic.obo ...
/home/laganiv/Desktop/projects/CausalEmbed/grn_crl/gCAL/gCRL/data/reference/ontologies/go-basic.obo: fmt(1.2) rel(2025-10-10) 42,666 Terms


In [20]:

# 3) Filter GO BP by size and (optionally) level
go_bp_filtered = filter_gene_sets_by_size_and_level(
    gene_sets=raw_go_bp,
    min_size=10,               # min genes per GO term
    max_size=200,              # max genes per GO term
    go_term_levels=go_term_levels,
    min_level=5,               # example: only level >= 4
    max_level=None,            # or e.g. 5 if you want 4–5 only
)
len(go_bp_filtered)

2684

In [21]:
# 4) Run ORA with universe = GRN genes (switch to False for library-wide universe)
ora_results = run_ora_for_clusters(
    cluster2genes=cluster2genes,
    gene_sets=go_bp_filtered,
    use_grn_universe=True,     # True: TF+targets universe; False: GO-library universe
    min_genes_in_cluster=5,
)

# 5) Save results
#ora_results.to_csv("tf_cluster_GO_BP_ORA_results.csv", index=False)
#print("Saved ORA results to tf_cluster_GO_BP_ORA_results.csv")

In [23]:
# For each cluster, show top 5 enriched GO BP terms
for cluster_id in ora_results['cluster_id'].unique():
    print(f"\nCluster {cluster_id} top GO BP terms:")
    top_terms = ora_results[ora_results['cluster_id'] == cluster_id].sort_values(by='pval').head(10)
    for _, row in top_terms.iterrows():
        print(f"  {row['go_term']} (P-value: {row['pval']:.4e}, adj p-value: {row['pval_adj']:.4e})")


Cluster 0 top GO BP terms:
  Positive Regulation of Catabolic Process (GO:0009896) (P-value: 1.5448e-01, adj p-value: 9.9995e-01)
  Positive Regulation of Biosynthetic Process (GO:0009891) (P-value: 1.5448e-01, adj p-value: 9.9995e-01)
  Regulation of Cytoskeleton Organization (GO:0051493) (P-value: 1.8197e-01, adj p-value: 9.9995e-01)
  Regulation of Transforming Growth Factor Beta Receptor Signaling Pathway (GO:0017015) (P-value: 1.9748e-01, adj p-value: 9.9995e-01)
  Regulation of Neuron Projection Development (GO:0010975) (P-value: 1.9748e-01, adj p-value: 9.9995e-01)
  Regulation of Cell Cycle Process (GO:0010564) (P-value: 2.1430e-01, adj p-value: 9.9995e-01)
  Regulation of Actin Cytoskeleton Organization (GO:0032956) (P-value: 2.1430e-01, adj p-value: 9.9995e-01)
  Cellular Response to Decreased Oxygen Levels (GO:0036294) (P-value: 2.3253e-01, adj p-value: 9.9995e-01)
  Regulation of Phosphorylation (GO:0042325) (P-value: 2.3253e-01, adj p-value: 9.9995e-01)
  Positive Regulat

Cluster 0 — Cytoskeletal Remodeling, Growth Control, and Oxygen Response Module
Biological interpretation

Cluster 0 is dominated by GO terms related to:

Cytoskeleton organization and actin remodeling

Cell cycle regulation

Positive regulation of catabolic & biosynthetic processes

Cellular response to hypoxia / decreased oxygen

TGF-β receptor signaling regulation

Functional meaning

This module likely groups TFs regulating structural cell remodeling and metabolic adaptation, coupled with growth control.
A plausible interpretation is that this TF module captures a stress-responsive proliferative program, involving:

Actin reorganization to reshape the cell

Adjusting metabolic fluxes (catabolism ↔ biosynthesis)

Hypoxia-driven gene expression

TGF-β regulatory interactions (typically anti-proliferative or pro-EMT)

Probable phenotype

A plasticity–stress response module, related to:

cytoskeletal tension changes

metabolic flexibility

responses to low oxygen

moderate growth regulation

Cluster 1 — Immune Signaling / MAPK Activation Module
Dominant processes

MAPK cascade

B-cell receptor signaling and T-cell activation

Protein processing / protein targeting

Cytokinesis and cell cycle control

Functional meaning

This module clearly clusters TFs involved in immune-related signaling pathways, especially:

MAPK / JNK signaling, a core inflammation/stress pathway

Adaptive immunity signals (TCR/BCR pathways)

Cytoskeletal & cytokinesis regulation, hinting at proliferation of immune-like cells or immune activation states

The presence of MAPK and antigen-receptor signaling together suggests a pro-inflammatory transcriptional program, even if the biological context is not immune-specific.

Probable phenotype

A pro-inflammatory, signal-transduction module, coordinating:

MAPK-dependent transcription

receptor-mediated immune signaling

protein processing steps essential for rapid signal propagation

Cluster 2 — Growth Suppression, TNF Response, Autophagy Module
Dominant processes

Negative regulation of cell growth

Response to TNF (inflammatory cytokine)

Regulation of serine/threonine kinase receptor signaling (TGF-β-like)

MAPK cascade

Autophagy activation

Regulation of immune-cell migration

Functional meaning

This cluster reflects a stress- or inflammation-induced growth inhibitory program. Key aspects:

TNF response (cytokine involved in apoptosis, inflammation)

Growth inhibition (GO:0030308)

Autophagy induction → a hallmark of metabolic stress

MAPK again suggesting stress signaling (JNK/p38 axis)

Enhanced migration pathways (mononuclear cell migration)

Probable phenotype

A cytokine-driven stress-response module coupling:

TNF → MAPK → autophagy

growth arrest

inflammation-induced motility signaling

This may represent TFs controlling catabolic, survival, and stress adaptation pathways.

Cluster 3 — Apoptosis, ECM Remodeling, Cytoskeleton Regulation Module
Dominant processes

Cytoskeleton / actin organization

Extracellular matrix (ECM) organization

Regulation of neuron apoptosis (both + and −)

Apoptotic signaling regulation

Regulation of EMT (epithelial-to-mesenchymal transition)

Cell cycle phase control

Functional meaning

This module appears linked to cell structural remodeling and fate decisions, particularly involving:

ECM → cytoskeleton → apoptosis → EMT
These processes often co-occur in tissue remodeling, cell migration, and differentiation, as well as in neuronal pruning and stress responses.

Probable phenotype

A cellular remodeling / apoptotic-fate module, integrating:

ECM and cytoskeletal changes

regulation of intrinsic apoptosis

EMT-related regulatory control

This may reflect TFs that tune the balance between:

survival vs apoptosis

epithelial vs mesenchymal phenotypes

stable morphology vs dynamic remodeling

Cluster 4 — Phosphorylation Control, Hypoxia Response, TGF-β Signaling Module
Dominant processes

Regulation of phosphorylation (highest enrichment)

Hypoxia response

TGF-β receptor signaling (positive and negative)

Cell cycle transitions

Negative regulation of translation

Phospholipid metabolic processes

Smooth muscle cell migration regulation

Transforming growth factor β response

Functional meaning

This cluster represents a kinase-centric, signaling-modulation module, focused on:

controlling phosphorylation states of proteins

metabolic cues under hypoxia

TGF-β pathways (canonical SMAD signaling)

translation regulation (a hallmark of stress and nutrient sensing)

The link between hypoxia and TGF-β is well-documented, e.g. in fibrosis, EMT, and stress adaptation.

Probable phenotype

A signal-modulation and hypoxia-TGFβ response module, integrating:

phosphorylation control

metabolic stress

anti-proliferative or pro-EMT TGF-β influences

migration-related signaling

This cluster may define a master regulatory layer for stress-dependent signaling tuning.

Cross-cluster summary

To help you use these interpretations when writing the GRN+CRL manuscript, here is how the clusters relate to each other:

Cluster	Dominant Biological Theme	Conceptual TF-module Role
0	Cytoskeleton, hypoxia, catabolism, TGF-β regulation	Structural remodeling + metabolic adaptation
1	MAPK, immune receptor signaling, protein processing	Inflammatory signaling + MAPK-driven activation
2	TNF response, autophagy, growth suppression	Inflammatory stress → growth arrest & autophagy
3	ECM, apoptosis, EMT, cytoskeleton	Remodeling + apoptotic fate regulation
4	Phosphorylation control, hypoxia, TGF-β	Signal fine-tuning under metabolic stress

One possible narrative linking them:

Clusters 1 & 2 reflect inflammatory and cytokine-driven signaling (MAPK / TNF).

Clusters 0, 3, 4 represent downstream cellular remodeling and stress-adaptation programs (cytoskeleton, apoptosis, hypoxia responses, TGF-β modulation).

Together, your GRN modules organize into a coherent landscape of stress → signaling → remodeling → fate decisions.

Cross-Cluster Causal Interpretation (Very Synthetic)

The five TF modules arrange into a coherent causal cascade of stress → signaling → remodeling → fate decisions:

Cluster 1 (MAPK / immune-signal activation)
acts as an upstream trigger, responding to receptor-mediated stimuli (MAPK, antigen-receptor signaling).
Likely causal influence: activates downstream stress-response and growth-control modules (Clusters 2, 4).

Cluster 2 (TNF response, growth arrest, autophagy)
represents a secondary stress-integration layer, translating inflammatory cues into growth suppression and autophagy.
Likely causal influence: modulates structural/adaptive modules (Clusters 0, 3) by altering metabolic state and catabolic flux.

Cluster 4 (phosphorylation control, hypoxia, TGF-β signaling)
functions as a signal-modulation hub, adjusting kinase activity, hypoxia responses, and TGF-β pathways in response to inputs from Clusters 1 and 2.
Likely causal influence: feeds into remodeling/fate modules (Cluster 3) and metabolic/cytoskeletal modules (Cluster 0).

Cluster 0 (cytoskeleton, metabolic adaptation, hypoxia response)
executes cellular adaptation, reshaping cytoskeleton and metabolic programs according to upstream signaling.
Likely causal influence: supports or restrains fate-decision programs in Cluster 3 through changes in actin tension and metabolic state.

Cluster 3 (ECM remodeling, apoptosis, EMT)
sits downstream, representing the final decision layer where cells commit to apoptosis, EMT, or structural remodeling.
Likely causal influence: minimal upstream feedback except via ECM–mechanotransduction back to Cluster 0.

One-sentence summary

Cluster 1 initiates signaling → Cluster 2 and 4 integrate stress and tune kinase pathways → Cluster 0 drives metabolic/cytoskeletal adaptation → Cluster 3 executes apoptotic or EMT-like fate decisions.

Let me know if you want this turned into a schematic figure (e.g., a causal flow diagram) for the manuscript.